In [0]:
import io, csv, zipfile, collections, time, requests

DAY = "20261010"   # one recent full day
URL = f"https://data.gdeltproject.org/events/{DAY}.export.CSV.zip"

# Oil producers, transit and conflict countries (GDELT location codes, FIPS)
GEO = {"IR","IZ","SA","KU","AE","QA","MU","BA","YM","IS","RS","UP","LY","NI","VE",
       "EG","DJ","ER","AG","AO","KZ","AJ","TU","SU","SO"}
# Same countries as actor codes (CAMEO)
ACTORS = {"IRN","IRQ","SAU","KWT","ARE","QAT","OMN","BHR","YEM","ISR","RUS","UKR","LBY",
          "NGA","VEN","EGY","DJI","ERI","DZA","AGO","KAZ","AZE","TUR","SDN","SOM"}
# Event types kept: cooperate/consult/diplomacy (03-06), yield incl. ceasefires (08),
# demand/disapprove/reject/threaten/protest/force/reduce relations incl. sanctions/
# coerce incl. seizures and blockades/assault/fight/mass violence (10-20)
ROOTS = {"03","04","05","06","08","10","11","12","13","14","15","16","17","18","19","20"}
OIL_WORDS = ("oil","crude","tanker","hormuz","refiner","pipeline","opec","petrol",
             "gasoline","diesel","lng","strait","shipping","red-sea","houthi","brent","barrel")

t0 = time.time()
r = requests.get(URL, timeout=300)
r.raise_for_status()
zipped_mb = len(r.content) / 1e6

total = kept = kept_5plus = by_word = 0
geo_count, root_count = collections.Counter(), collections.Counter()
samples = []

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    with z.open(z.namelist()[0]) as f:
        reader = csv.reader(io.TextIOWrapper(f, encoding="utf-8", errors="replace"), delimiter="\t")
        for row in reader:
            total += 1
            if len(row) < 58:
                continue
            a1c, a2c = row[7], row[17]
            root, mentions = row[28], int(row[31] or 0)
            geo = row[51]
            url = row[57].lower()
            place_ok = geo in GEO or a1c in ACTORS or a2c in ACTORS
            word_ok = any(w in url for w in OIL_WORDS)
            if (place_ok and root in ROOTS) or word_ok:
                kept += 1
                by_word += word_ok
                kept_5plus += mentions >= 5
                geo_count[geo] += 1
                root_count[root] += 1
                if word_ok and len(samples) < 6 and any(w in url for w in ("hormuz","tanker","refiner")):
                    samples.append((row[6][:20], row[16][:20], root, mentions, row[50][:35], row[57][:90]))

print(f"Day {DAY}: zipped {zipped_mb:.1f} MB, download+parse {time.time()-t0:.0f}s")
print(f"All events: {total:,}")
print(f"Kept by filter: {kept:,}  ({kept/total:.1%})  | with oil word in link: {by_word:,}  | 5+ mentions: {kept_5plus:,}")
print("\nTop locations (kept):", geo_count.most_common(10))
print("Top event types (kept):", root_count.most_common(10))
print("\nSample oil-related events:")
for s in samples:
    print(" ", s)